# 02 - Embedding: vektor dokumen dan query dengan congen-indobert-base

Alur: set jumlah thread → muat Dokumen dan Query → muat model pada revision yang dicatat → periksa loop encode terhadap `model.encode` pada 100 teks → encode semua dokumen dan query di GPU → simpan vektor dan Set embedding → catat Lingkungan.

Keluaran:
- `data/embeddings/<embedding_id>/doc_vectors.npy`, `query_vectors.npy` — float32, norma L2 = 1
- `data/embeddings/<embedding_id>/doc_ids.parquet`, `query_ids.parquet` — urutan baris vektor
- `data/embeddings/<embedding_id>/embedding_set.json` — konfigurasi, revision, batch akhir, hasil pemeriksaan encode
- `outputs/tuning/environment_<env_id>.json` — Lingkungan (K8)

Prasyarat: `01_preprocessing` sudah dijalankan; instance Vast.ai dengan GPU (K6); `pip install -r requirements.txt`.

Keputusan: K1 (003a), K6 (002a), K8 (004a), K9 (003a).

In [ ]:
from pathlib import Path

# Keluaran 01_preprocessing: documents/queries.parquet dan metadata.json
PROCESSED_DIR = Path("../data/processed")
# Folder induk Set embedding; satu subfolder per embedding_id
EMBEDDINGS_DIR = Path("../data/embeddings")
# Folder catatan Lingkungan (K8)
TUNING_DIR = Path("../outputs/tuning")
# Root project, dipakai untuk membaca total dan sisa disk
PROJECT_ROOT = Path("..")
# Nama notebook di catatan Lingkungan
NOTEBOOK_NAME = "02_embedding"

# Model embedding yang dikunci Arya
MODEL_NAME = "LazarusNLP/congen-indobert-base"
# Revision yang diunduh; commit hash hasilnya dicatat di Set embedding
MODEL_REVISION = "main"
# Panjang token maksimum untuk dokumen dan query (K1)
MAX_SEQ_LENGTH = 32
# Dimensi vektor keluaran model
EMBEDDING_DIM = 768
# Ukuran batch encode, dicoba mulai 1024; nilai akhir dicatat di Set embedding (K1)
BATCH_SIZE = 1024
# Encode di GPU dengan presisi fp32 (K1)
DEVICE = "cuda"
PRECISION = "fp32"
# Urutan modul model yang dilewati loop encode (K1)
EXPECTED_MODULES = ["Transformer", "Pooling", "Dense"]
# Aturan teks yang di-embed, dicatat di Set embedding (K1)
DOCUMENT_TEXT_RULE = 'title + " " + text'
QUERY_TEXT_RULE = "text apa adanya"
NORMALIZATION_RULE = "L2, F.normalize sekali saat encode"

# Jumlah teks sampel pemeriksaan loop encode terhadap model.encode (K1)
ENCODE_CHECK_SAMPLES = 100
# Selisih mutlak maksimum per elemen yang diizinkan pada pemeriksaan encode (K1)
ENCODE_CHECK_TOLERANCE = 1e-5
# Batas |norma L2 - 1| tiap vektor tersimpan
NORM_TOLERANCE = 1e-5
# Panjang embedding_id: awal hash SHA-256 konfigurasi, dalam karakter heksadesimal
EMBEDDING_ID_LENGTH = 16
# Cetak kemajuan encode setiap sekian batch
PROGRESS_EVERY = 100

# Folder induk cgroup: v2 langsung di sini, v1 di subfolder cpu/ dan memory/ (K6, K8, 005a)
CGROUP_DIR = Path("/sys/fs/cgroup")
# Batas RAM cgroup v1 sebesar ini atau lebih dianggap tanpa batas, sama seperti "max" di v2 (005a)
MEMORY_UNLIMITED_FROM = 2**62
# Sumber model CPU dan flag AVX (K8)
CPUINFO_PATH = Path("/proc/cpuinfo")
# Batas atas jumlah thread (K6)
MAX_THREADS = 24
# Paket yang versinya ikut membentuk env_id; daftar ini sama di semua notebook (K8)
ENVIRONMENT_LIBRARIES = [
    "torch",
    "faiss-cpu",
    "sentence-transformers",
    "transformers",
    "datasets",
    "numpy",
    "pandas",
    "pyarrow",
]
# Panjang env_id: awal hash SHA-256 field statis Lingkungan, dalam karakter heksadesimal
ENV_ID_LENGTH = 16

## 1. Thread

Jumlah thread = ⌊min(jatah CPU, 24)⌋ (K6). Jatah CPU = kuota cgroup, dibatasi jumlah CPU di `sched_getaffinity`; `os.cpu_count()` tidak dipakai karena melaporkan total mesin. Kuota dibaca dari cgroup v2 (`cpu.max`) atau v1 (`cpu.cfs_quota_us` / `cpu.cfs_period_us`) (005a). Host yang punya berkas v2 dan v1 sekaligus menghentikan notebook karena urutan deteksinya belum diputuskan (H18). Pembacaan ini hanya berjalan di Linux (H12).

In [ ]:
import math
import os

import faiss
import torch


def parse_cpu_max(text: str) -> float | None:
    """Ubah isi cgroup v2 `cpu.max` menjadi jumlah CPU.

    Args:
        text: Isi berkas, contoh "400000 100000" (4 CPU) atau "max 100000" (tanpa batas).

    Returns:
        Kuota dalam satuan CPU, atau None kalau tanpa batas.
    """
    quota, period = text.split()
    return None if quota == "max" else int(quota) / int(period)


def parse_cfs_quota(quota_text: str, period_text: str) -> float | None:
    """Ubah isi cgroup v1 `cpu.cfs_quota_us` dan `cpu.cfs_period_us` menjadi jumlah CPU.

    Args:
        quota_text: Isi cpu.cfs_quota_us, contoh "400000", atau "-1" (tanpa batas).
        period_text: Isi cpu.cfs_period_us, contoh "100000".

    Returns:
        Kuota dalam satuan CPU, atau None kalau tanpa batas.
    """
    quota = int(quota_text.strip())
    return None if quota < 0 else quota / int(period_text.strip())


def fetch_cgroup_cpu_quota(cgroup_dir: Path) -> float | None:
    """Baca kuota CPU dari cgroup v2 atau v1.

    Args:
        cgroup_dir: Folder induk cgroup.

    Returns:
        Kuota dalam satuan CPU, atau None kalau tanpa batas.

    Raises:
        ValueError: Berkas v2 dan v1 sama-sama ada (host hybrid, urutan deteksi belum diputuskan, H18).
        FileNotFoundError: Berkas v2 maupun v1 tidak ada.
    """
    v2_path = cgroup_dir / "cpu.max"
    v1_quota_path = cgroup_dir / "cpu" / "cpu.cfs_quota_us"
    v1_period_path = cgroup_dir / "cpu" / "cpu.cfs_period_us"
    has_v2 = v2_path.is_file()
    has_v1 = v1_quota_path.is_file() and v1_period_path.is_file()
    if has_v2 and has_v1:
        raise ValueError(
            f"{v2_path} dan {v1_quota_path} sama-sama ada; urutan deteksi host hybrid belum diputuskan (H18)"
        )
    if has_v2:
        return parse_cpu_max(v2_path.read_text(encoding="utf-8"))
    if has_v1:
        return parse_cfs_quota(v1_quota_path.read_text(encoding="utf-8"), v1_period_path.read_text(encoding="utf-8"))
    raise FileNotFoundError(f"Tidak ada {v2_path} maupun {v1_quota_path}; butuh cgroup v1 atau v2 di Linux (H12)")


def fetch_cpu_quota(cgroup_dir: Path) -> float:
    """Baca jatah CPU efektif instance.

    Args:
        cgroup_dir: Folder induk cgroup.

    Returns:
        min(kuota cgroup, jumlah CPU sched_getaffinity), atau jumlah CPU affinity kalau kuota tanpa batas.

    Raises:
        OSError: Bukan Linux (os.sched_getaffinity tidak ada).
    """
    if not hasattr(os, "sched_getaffinity"):
        raise OSError("os.sched_getaffinity tidak tersedia; jatah CPU hanya bisa dibaca di Linux (H12)")
    cgroup_quota = fetch_cgroup_cpu_quota(cgroup_dir)
    affinity_count = float(len(os.sched_getaffinity(0)))
    return affinity_count if cgroup_quota is None else min(cgroup_quota, affinity_count)


def compute_thread_count(cpu_quota: float, max_threads: int) -> int:
    """Hitung jumlah thread dari jatah CPU (K6).

    Args:
        cpu_quota: Jatah CPU efektif.
        max_threads: Batas atas thread.

    Returns:
        ⌊min(cpu_quota, max_threads)⌋, paling sedikit 1.
    """
    return max(1, math.floor(min(cpu_quota, max_threads)))


def set_num_threads(count: int) -> None:
    """Set jumlah thread FAISS dan torch.

    Args:
        count: Jumlah thread.
    """
    faiss.omp_set_num_threads(count)
    torch.set_num_threads(count)


cpu_quota = fetch_cpu_quota(CGROUP_DIR)
thread_count = compute_thread_count(cpu_quota, MAX_THREADS)
set_num_threads(thread_count)
print({"cpu_quota": cpu_quota, "thread_count": thread_count})

## 2. Muat Dokumen dan Query

Teks dokumen = `title + " " + text`; query dipakai apa adanya (K1). Judul atau teks yang kosong menghentikan notebook karena aturan K1 tidak menentukan penggantinya.

In [ ]:
import json

import pandas as pd


def load_parquet(path: Path) -> pd.DataFrame:
    """Baca satu tabel Parquet.

    Args:
        path: Berkas Parquet.

    Returns:
        Isi tabel.

    Raises:
        FileNotFoundError: Berkas tidak ada.
    """
    if not path.is_file():
        raise FileNotFoundError(f"Berkas tidak ditemukan: {path}")
    return pd.read_parquet(path)


def load_json(path: Path) -> dict[str, object]:
    """Baca satu berkas JSON.

    Args:
        path: Berkas JSON.

    Returns:
        Isi berkas.

    Raises:
        FileNotFoundError: Berkas tidak ada.
    """
    if not path.is_file():
        raise FileNotFoundError(f"Berkas tidak ditemukan: {path}")
    with path.open(encoding="utf-8") as file:
        return json.load(file)


def validate_text_columns(frame: pd.DataFrame, columns: list[str], name: str) -> None:
    """Pastikan kolom teks terisi di semua baris.

    Args:
        frame: Tabel yang diperiksa.
        columns: Kolom teks.
        name: Nama tabel untuk pesan error.

    Raises:
        ValueError: Ada nilai kosong (null).
    """
    empty = {column: int(frame[column].isna().sum()) for column in columns if frame[column].isna().any()}
    if empty:
        raise ValueError(f"{name}: kolom teks kosong {empty}")


def to_document_texts(documents: pd.DataFrame) -> list[str]:
    """Bentuk teks dokumen yang di-embed (K1).

    Args:
        documents: Tabel Dokumen dengan kolom title dan text.

    Returns:
        title + " " + text per dokumen, dalam urutan baris tabel.
    """
    return (documents["title"] + " " + documents["text"]).tolist()


documents = load_parquet(PROCESSED_DIR / "documents.parquet")
queries = load_parquet(PROCESSED_DIR / "queries.parquet")
processed_metadata = load_json(PROCESSED_DIR / "metadata.json")
validate_text_columns(documents, ["title", "text"], "documents")
validate_text_columns(queries, ["text"], "queries")
document_texts = to_document_texts(documents)
query_texts = queries["text"].tolist()
dataset_revisions = {name: source["revision"] for name, source in processed_metadata["sources"].items()}
print({"documents": len(document_texts), "queries": len(query_texts), "dataset_revisions": dataset_revisions})

## 3. Muat model

Model diunduh lewat `snapshot_download` supaya commit hash snapshot yang benar-benar dimuat bisa dicatat (K1). sentence-transformers hanya memuat model; encode dikerjakan loop PyTorch di tahap berikutnya.

In [ ]:
import re

from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer

# Commit hash Hugging Face: 40 karakter heksadesimal, contoh 0123abcd...89ef
COMMIT_HASH_PATTERN = re.compile(r"[0-9a-f]{40}")


def load_embedding_model(name: str, revision: str, device: str) -> tuple[SentenceTransformer, str]:
    """Unduh dan muat model pada revision tertentu dalam fp32.

    Args:
        name: Repo model Hugging Face.
        revision: Branch, tag, atau commit yang diminta.
        device: Device model.

    Returns:
        Model dalam mode eval fp32 dan commit hash snapshot yang dimuat.

    Raises:
        ValueError: Nama folder snapshot bukan commit hash.
    """
    snapshot_path = Path(snapshot_download(repo_id=name, revision=revision))
    if COMMIT_HASH_PATTERN.fullmatch(snapshot_path.name) is None:
        raise ValueError(f"Folder snapshot bukan commit hash: {snapshot_path}")
    model = SentenceTransformer(str(snapshot_path), device=device)
    return model.float().eval(), snapshot_path.name


def validate_model_setup(model: SentenceTransformer, expected_modules: list[str], max_seq_length: int) -> None:
    """Pastikan susunan modul dan panjang token model sesuai K1.

    Args:
        model: Model yang dimuat.
        expected_modules: Nama kelas modul sesuai urutan.
        max_seq_length: Panjang token maksimum yang diharapkan.

    Raises:
        ValueError: Susunan modul, aktivasi Dense, atau max_seq_length berbeda.
    """
    modules = [type(module).__name__ for module in model]
    if modules != expected_modules:
        raise ValueError(f"Modul model {modules}, seharusnya {expected_modules}")
    activation = type(model[2].activation_function).__name__
    if activation != "Tanh":
        raise ValueError(f"Aktivasi Dense {activation}, seharusnya Tanh")
    if model.max_seq_length != max_seq_length:
        raise ValueError(f"max_seq_length model {model.max_seq_length}, seharusnya {max_seq_length}")


if not torch.cuda.is_available():
    raise RuntimeError("CUDA tidak tersedia; embedding dijalankan di GPU instance Vast.ai (K6)")
model, model_revision = load_embedding_model(MODEL_NAME, MODEL_REVISION, DEVICE)
validate_model_setup(model, EXPECTED_MODULES, MAX_SEQ_LENGTH)
print({"model": MODEL_NAME, "model_revision": model_revision})

## 4. Pemeriksaan loop encode

Loop encode sendiri dibandingkan dengan `model.encode` pada 100 teks dokumen pertama; keduanya sudah dinormalisasi L2. Selisih mutlak maksimum per elemen harus ≤ 1e-5 (K1). Kalau gagal, notebook berhenti dan toleransinya tidak dilonggarkan.

In [ ]:
import numpy as np
import torch.nn.functional as F


def encode_texts(
    model: SentenceTransformer, texts: list[str], max_length: int, batch_size: int, device: str
) -> np.ndarray:
    """Encode teks dengan loop PyTorch: Transformer → Pooling → Dense+Tanh → normalisasi L2 (K1).

    Args:
        model: Model sentence-transformers yang sudah dimuat.
        texts: Teks yang di-encode.
        max_length: Panjang token maksimum saat tokenisasi.
        batch_size: Jumlah teks per batch.
        device: Device tempat tensor diproses.

    Returns:
        Matriks float32 berukuran (len(texts), dimensi model), setiap baris bernorma 1.
    """
    vectors: np.ndarray | None = None
    batch_total = math.ceil(len(texts) / batch_size)
    with torch.inference_mode():
        for batch_index, start in enumerate(range(0, len(texts), batch_size)):
            batch = texts[start : start + batch_size]
            tokens = model.tokenizer(
                batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt"
            )
            features = {key: value.to(device) for key, value in tokens.items()}
            for module in (model[0], model[1], model[2]):
                features = module(features)
            embeddings = F.normalize(features["sentence_embedding"], p=2, dim=1).float().cpu().numpy()
            if vectors is None:
                vectors = np.empty((len(texts), embeddings.shape[1]), dtype=np.float32)
            vectors[start : start + len(batch)] = embeddings
            if (batch_index + 1) % PROGRESS_EVERY == 0:
                print(f"batch {batch_index + 1}/{batch_total}")
    return vectors


def compare_with_model_encode(
    model: SentenceTransformer, texts: list[str], max_length: int, batch_size: int, device: str
) -> float:
    """Hitung selisih mutlak maksimum antara loop encode dan model.encode.

    Args:
        model: Model sentence-transformers.
        texts: Teks sampel.
        max_length: Panjang token maksimum loop encode.
        batch_size: Ukuran batch kedua cara encode.
        device: Device encode.

    Returns:
        max |v_loop - v_model.encode| atas semua elemen; kedua vektor sudah ternormalisasi.
    """
    loop_vectors = encode_texts(model, texts, max_length, batch_size, device)
    reference = model.encode(
        texts, batch_size=batch_size, convert_to_numpy=True, normalize_embeddings=True, device=device
    )
    return float(np.abs(loop_vectors - reference.astype(np.float32)).max())


def validate_encode_match(max_abs_diff: float, tolerance: float) -> None:
    """Pastikan loop encode sama dengan model.encode dalam toleransi K1.

    Args:
        max_abs_diff: Selisih mutlak maksimum per elemen.
        tolerance: Batas yang diizinkan.

    Raises:
        ValueError: Selisih melebihi toleransi.
    """
    if max_abs_diff > tolerance:
        raise ValueError(
            f"Loop encode berbeda dari model.encode: selisih maksimum {max_abs_diff:.3e} > {tolerance:.0e}"
        )


encode_check_texts = document_texts[:ENCODE_CHECK_SAMPLES]
encode_check_diff = compare_with_model_encode(model, encode_check_texts, MAX_SEQ_LENGTH, BATCH_SIZE, DEVICE)
validate_encode_match(encode_check_diff, ENCODE_CHECK_TOLERANCE)
print({"samples": len(encode_check_texts), "max_abs_diff": encode_check_diff})

## 5. Encode dokumen dan query

Semua dokumen dan query di-encode sekali (K1). Vektor yang tersimpan wajib bernorma 1 karena penalti jarak 2 pada metrik #8 hanya sah untuk vektor bernorma 1.

In [ ]:
def validate_vectors(vectors: np.ndarray, expected_rows: int, dim: int, tolerance: float, name: str) -> None:
    """Pastikan bentuk, tipe, dan norma vektor sesuai model data.

    Args:
        vectors: Matriks vektor.
        expected_rows: Jumlah baris yang diharapkan.
        dim: Dimensi vektor.
        tolerance: Batas |norma L2 - 1|.
        name: Nama matriks untuk pesan error.

    Raises:
        ValueError: Bentuk, tipe, atau norma tidak sesuai.
    """
    if vectors.shape != (expected_rows, dim) or vectors.dtype != np.float32:
        raise ValueError(f"{name}: bentuk {vectors.shape} {vectors.dtype}, seharusnya ({expected_rows}, {dim}) float32")
    norm_error = float(np.abs(np.linalg.norm(vectors, axis=1) - 1.0).max())
    if norm_error > tolerance:
        raise ValueError(f"{name}: |norma - 1| maksimum {norm_error:.3e} > {tolerance:.0e}")


torch.cuda.reset_peak_memory_stats()
doc_vectors = encode_texts(model, document_texts, MAX_SEQ_LENGTH, BATCH_SIZE, DEVICE)
query_vectors = encode_texts(model, query_texts, MAX_SEQ_LENGTH, BATCH_SIZE, DEVICE)
validate_vectors(doc_vectors, len(documents), EMBEDDING_DIM, NORM_TOLERANCE, "doc_vectors")
validate_vectors(query_vectors, len(queries), EMBEDDING_DIM, NORM_TOLERANCE, "query_vectors")
print({"doc_vectors": doc_vectors.shape, "query_vectors": query_vectors.shape})

## 6. Set embedding dan simpan

embedding_id = awal hash SHA-256 dari konfigurasi Set embedding (model, revision model dan dataset, panjang token, aturan teks, presisi, normalisasi, dimensi). Ukuran batch dan hasil pemeriksaan encode dicatat tetapi tidak ikut membentuk embedding_id. Folder embedding_id yang sudah ada tidak ditimpa.

In [ ]:
import hashlib
from datetime import datetime, timezone


def build_embedding_config(model_revision: str, dataset_revisions: dict[str, str]) -> dict[str, object]:
    """Susun konfigurasi yang menentukan isi vektor (K1).

    Args:
        model_revision: Commit hash model yang dimuat.
        dataset_revisions: Commit hash dataset per tabel sumber.

    Returns:
        Konfigurasi Set embedding.
    """
    return {
        "model": MODEL_NAME,
        "model_revision": model_revision,
        "dataset_revisions": dataset_revisions,
        "max_seq_length": MAX_SEQ_LENGTH,
        "document_text": DOCUMENT_TEXT_RULE,
        "query_text": QUERY_TEXT_RULE,
        "modules": EXPECTED_MODULES,
        "precision": PRECISION,
        "normalization": NORMALIZATION_RULE,
        "dim": EMBEDDING_DIM,
    }


def compute_embedding_id(config: dict[str, object], length: int) -> str:
    """Hitung embedding_id dari konfigurasi.

    Args:
        config: Konfigurasi Set embedding.
        length: Jumlah karakter heksadesimal yang diambil dari awal hash.

    Returns:
        Awal SHA-256 dari JSON konfigurasi berkunci terurut.
    """
    canonical = json.dumps(config, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:length]


def save_embeddings(
    target_dir: Path,
    vectors: dict[str, np.ndarray],
    ids: dict[str, pd.DataFrame],
    embedding_set: dict[str, object],
) -> None:
    """Simpan vektor, urutan id, dan Set embedding ke folder baru.

    Args:
        target_dir: Folder `data/embeddings/<embedding_id>`; tidak boleh sudah ada.
        vectors: Nama berkas tanpa ekstensi ke matriks float32, disimpan sebagai `.npy`.
        ids: Nama berkas tanpa ekstensi ke tabel id, disimpan sebagai Parquet.
        embedding_set: Isi embedding_set.json.

    Raises:
        FileExistsError: Folder embedding_id sudah ada.
    """
    target_dir.mkdir(parents=True, exist_ok=False)
    for name, matrix in vectors.items():
        np.save(target_dir / f"{name}.npy", matrix)
    for name, table in ids.items():
        table.to_parquet(target_dir / f"{name}.parquet", index=False)
    with (target_dir / "embedding_set.json").open("w", encoding="utf-8", newline="\n") as file:
        json.dump(embedding_set, file, ensure_ascii=False, indent=2)


embedding_config = build_embedding_config(model_revision, dataset_revisions)
embedding_id = compute_embedding_id(embedding_config, EMBEDDING_ID_LENGTH)
embedding_set = {
    "embedding_id": embedding_id,
    "config": embedding_config,
    "batch_size": BATCH_SIZE,
    "encode_check": {
        "samples": len(encode_check_texts),
        "max_abs_diff": encode_check_diff,
        "tolerance": ENCODE_CHECK_TOLERANCE,
    },
    "counts": {"documents": len(doc_vectors), "queries": len(query_vectors)},
    # Cap waktu UTC format ISO 8601, contoh 2026-10-02T08:30:00+00:00
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
save_embeddings(
    EMBEDDINGS_DIR / embedding_id,
    {"doc_vectors": doc_vectors, "query_vectors": query_vectors},
    {"doc_ids": documents[["doc_id"]], "query_ids": queries[["query_id"]]},
    embedding_set,
)
print({"embedding_id": embedding_id, "folder": str(EMBEDDINGS_DIR / embedding_id)})

## 7. Lingkungan

env_id = awal hash SHA-256 dari semua field statis ditambah hostname; field dinamis (puncak RAM dan VRAM, sisa disk, timestamp) dicatat tetapi tidak masuk hash (K8, 004a). Satu berkas per env_id; setiap notebook menambahkan catatan field dinamisnya sendiri ke berkas itu.

In [ ]:
import importlib.metadata
import platform
import resource
import shutil
import socket
import subprocess


def parse_nvidia_smi(text: str) -> list[dict[str, str | int]]:
    """Ubah keluaran nvidia-smi menjadi daftar GPU.

    Args:
        text: Keluaran `nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader,nounits`,
            satu baris per GPU, contoh "NVIDIA GeForce RTX 3090, 24576, 550.54.14".

    Returns:
        Per GPU: nama, VRAM total (MiB), versi driver.
    """
    gpus = []
    for line in text.strip().splitlines():
        name, memory_total, driver = (part.strip() for part in line.rsplit(",", 2))
        gpus.append({"name": name, "memory_total_mib": int(memory_total), "driver": driver})
    return gpus


def fetch_gpu_info() -> list[dict[str, str | int]]:
    """Baca model GPU, VRAM total, dan driver lewat nvidia-smi.

    Returns:
        Daftar GPU.
    """
    command = ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"]
    return parse_nvidia_smi(subprocess.run(command, capture_output=True, text=True, check=True).stdout)


def parse_cpuinfo(text: str) -> dict[str, str | bool]:
    """Ambil model CPU dan flag AVX dari isi /proc/cpuinfo.

    Args:
        text: Isi /proc/cpuinfo.

    Returns:
        Model CPU, ada tidaknya flag avx2 dan avx512f.

    Raises:
        ValueError: Baris "model name" atau "flags" tidak ada.
    """
    fields = {}
    for line in text.splitlines():
        key, _, value = line.partition(":")
        fields.setdefault(key.strip(), value.strip())
    if "model name" not in fields or "flags" not in fields:
        raise ValueError("/proc/cpuinfo tidak memuat 'model name' atau 'flags'")
    flags = set(fields["flags"].split())
    return {"model": fields["model name"], "avx2": "avx2" in flags, "avx512f": "avx512f" in flags}


def parse_memory_max(text: str) -> int | None:
    """Ubah isi cgroup v2 `memory.max` menjadi byte.

    Args:
        text: Isi berkas, contoh "34359738368" atau "max".

    Returns:
        Jatah RAM dalam byte, atau None kalau tanpa batas.
    """
    value = text.strip()
    return None if value == "max" else int(value)


def parse_memory_limit(text: str, unlimited_from: int) -> int | None:
    """Ubah isi cgroup v1 `memory.limit_in_bytes` menjadi byte.

    Args:
        text: Isi berkas, contoh "34359738368", atau angka sekitar 2^63 untuk tanpa batas.
        unlimited_from: Nilai mulai dari angka ini dianggap tanpa batas.

    Returns:
        Jatah RAM dalam byte, atau None kalau tanpa batas.
    """
    value = int(text.strip())
    return None if value >= unlimited_from else value


def fetch_memory_quota(cgroup_dir: Path, unlimited_from: int) -> int | None:
    """Baca jatah RAM dari cgroup v2 atau v1.

    Args:
        cgroup_dir: Folder induk cgroup.
        unlimited_from: Batas v1 yang dianggap tanpa batas.

    Returns:
        Jatah RAM dalam byte, atau None kalau tanpa batas.

    Raises:
        ValueError: Berkas v2 dan v1 sama-sama ada (host hybrid, urutan deteksi belum diputuskan, H18).
        FileNotFoundError: Berkas v2 maupun v1 tidak ada.
    """
    v2_path = cgroup_dir / "memory.max"
    v1_path = cgroup_dir / "memory" / "memory.limit_in_bytes"
    has_v2, has_v1 = v2_path.is_file(), v1_path.is_file()
    if has_v2 and has_v1:
        raise ValueError(f"{v2_path} dan {v1_path} sama-sama ada; urutan deteksi host hybrid belum diputuskan (H18)")
    if has_v2:
        return parse_memory_max(v2_path.read_text(encoding="utf-8"))
    if has_v1:
        return parse_memory_limit(v1_path.read_text(encoding="utf-8"), unlimited_from)
    raise FileNotFoundError(f"Tidak ada {v2_path} maupun {v1_path}; butuh cgroup v1 atau v2 di Linux (H12)")


def fetch_library_versions(names: list[str]) -> dict[str, str]:
    """Baca versi paket terpasang tanpa meng-import paketnya.

    Args:
        names: Nama distribusi paket.

    Returns:
        Nama paket ke versi.
    """
    return {name: importlib.metadata.version(name) for name in names}


def collect_environment(
    cpu_quota: float, revisions: dict[str, object], project_root: Path, cgroup_dir: Path, cpuinfo_path: Path
) -> dict[str, dict[str, object]]:
    """Kumpulkan field statis dan dinamis Lingkungan (K8, 004a).

    Args:
        cpu_quota: Jatah CPU efektif dari fetch_cpu_quota.
        revisions: Revision model dan dataset.
        project_root: Root project untuk ukuran disk.
        cgroup_dir: Folder cgroup v2.
        cpuinfo_path: Berkas /proc/cpuinfo.

    Returns:
        {"static": field yang membentuk env_id, "dynamic": field yang hanya dicatat}.
    """
    disk = shutil.disk_usage(project_root)
    static = {
        "gpus": fetch_gpu_info(),
        "cuda": torch.version.cuda,
        "cpu": parse_cpuinfo(cpuinfo_path.read_text(encoding="utf-8")),
        "cpu_quota": cpu_quota,
        "cpu_total": os.cpu_count(),
        "memory_quota_bytes": fetch_memory_quota(cgroup_dir, MEMORY_UNLIMITED_FROM),
        "disk_total_bytes": disk.total,
        "threads_faiss": faiss.omp_get_max_threads(),
        "threads_torch": torch.get_num_threads(),
        "python": platform.python_version(),
        "libraries": fetch_library_versions(ENVIRONMENT_LIBRARIES),
        "revisions": revisions,
        "hostname": socket.gethostname(),
    }
    dynamic = {
        # ru_maxrss di Linux dalam KiB
        "peak_ram_bytes": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024,
        "peak_vram_bytes": torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None,
        "disk_free_bytes": disk.free,
        # Cap waktu UTC format ISO 8601, contoh 2026-10-02T08:30:00+00:00
        "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    return {"static": static, "dynamic": dynamic}


def compute_env_id(static: dict[str, object], length: int) -> str:
    """Hitung env_id dari field statis Lingkungan.

    Args:
        static: Field statis, termasuk hostname.
        length: Jumlah karakter heksadesimal yang diambil dari awal hash.

    Returns:
        Awal SHA-256 dari JSON field statis berkunci terurut.
    """
    canonical = json.dumps(static, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:length]


def save_environment(environment: dict[str, dict[str, object]], env_id: str, notebook: str, tuning_dir: Path) -> Path:
    """Tambahkan catatan Lingkungan ke `environment_<env_id>.json`.

    Args:
        environment: Hasil collect_environment.
        env_id: Hasil compute_env_id.
        notebook: Nama notebook pencatat.
        tuning_dir: Folder outputs/tuning.

    Returns:
        Path berkas Lingkungan.

    Raises:
        ValueError: Berkas env_id yang sama sudah ada tetapi field statisnya berbeda.
    """
    path = tuning_dir / f"environment_{env_id}.json"
    record = {"notebook": notebook, **environment["dynamic"]}
    if path.is_file():
        with path.open(encoding="utf-8") as file:
            content = json.load(file)
        if content["static"] != json.loads(json.dumps(environment["static"])):
            raise ValueError(f"{path}: field statis berbeda untuk env_id yang sama")
        content["records"].append(record)
    else:
        content = {"env_id": env_id, "static": environment["static"], "records": [record]}
    tuning_dir.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="\n") as file:
        json.dump(content, file, ensure_ascii=False, indent=2)
    return path


environment = collect_environment(
    cpu_quota, {"model": model_revision, "datasets": dataset_revisions}, PROJECT_ROOT, CGROUP_DIR, CPUINFO_PATH
)
env_id = compute_env_id(environment["static"], ENV_ID_LENGTH)
environment_path = save_environment(environment, env_id, NOTEBOOK_NAME, TUNING_DIR)
print({"env_id": env_id, "path": str(environment_path), **environment["dynamic"]})